In [0]:
%pip install yfinance

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS oil_stock")
spark.sql("USE CATALOG oil_stock")

In [0]:
import yfinance as yf
import pandas as pd

tickers = ["EQNR.OL", "BZ=F"]

frames = []

for ticker in tickers:
    df = yf.download(
        ticker,
        period="10y",
        interval="1d",
        auto_adjust=False
    )

    df = df.reset_index()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df["symbol"] = ticker

    frames.append(df)

daily = pd.concat(frames, ignore_index=True)

display(daily)

In [0]:
from pyspark.sql import functions as F

bronze_daily_df = (
    spark.createDataFrame(daily)
    .select(
        F.col("Date").alias("trading_date"),
        F.col("symbol"),
        F.col("Open").alias("open"),
        F.col("High").alias("high"),
        F.col("Low").alias("low"),
        F.col("Close").alias("close"),
        F.col("Volume").alias("volume")
    )
    .withColumn(
        "currency",
        F.when(F.col("symbol") == "EQNR.OL", "NOK")
         .when(F.col("symbol") == "BZ=F", "USD")
    )
    .withColumn("source", F.lit("yfinance"))
    .withColumn("ingested_at", F.current_timestamp())
)

display(bronze_daily_df)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

bronze_daily_df.write.mode("overwrite").saveAsTable("bronze.market_daily")